# 04-锁存器、寄存器与内存

> 组合逻辑只能根据当前输入计算结果；本篇引入反馈和时钟，让电路能够“记住”状态，并从 1 位存储单元扩展到寄存器和内存。

## 1. 本篇学习范围

本篇对应课程视频 **41—52**，主题是锁存器、寄存器和内存：

- 41_计算器和计算机的区别
- 42_用电去存储电信号
- 43_锁存器
- 44_带边缘触发的锁存器
- 45_8位寄存器的抽象
- 46_输出使能开关
- 47_系统自带的驱动器
- 48_寄存器输入和输出线路的计算
- 49_并行转串行的原理
- 50_构建一个内存单元格
- 51_构建一个矩阵内存
- 52_实际内存的结构

本篇的核心问题是：**如何保存一个比特，如何按时钟更新它，如何用地址找到大量比特。**

## 2. 计算器和计算机的区别

计算器通常面向少量固定运算：输入数字，执行运算，显示结果。计算机则把数据、指令和中间结果放进存储器，由控制逻辑按步骤读取和执行。

| 对比点 | 计算器 | 计算机 |
|---|---|---|
| 用途 | 面向特定计算 | 可通过程序执行多种任务 |
| 状态 | 可能只保留当前结果 | 需要保存指令、数据和中间状态 |
| 控制 | 功能相对固定 | 控制器按时序驱动数据通路 |
| 存储 | 存储需求较少 | 需要寄存器和更大容量的内存 |

因此，前面学到的加法器和 ALU 还不够。要成为能执行程序的计算机，电路必须具有**状态**：在这一个时刻写入的数据，下一时刻仍然可以读出来。

## 3. 用电信号存储 0 和 1

数字存储的最小单位是比特（bit），它只有 0 和 1 两种状态。电路需要满足两个条件：

1. 输入一个写入信号时，能够把状态设置成 0 或 1；
2. 没有写入时，输出仍保持原来的状态。

单纯把一个逻辑门的输出接到下一个逻辑门，只能传递当前结果；要“记住”状态，需要把输出通过反馈重新送回输入，形成稳定的两个状态。

### 3.1 组合逻辑和时序逻辑

- **组合逻辑：**输出只由当前输入决定，例如 AND、加法器。
- **时序逻辑：**输出由当前输入和过去保存的状态共同决定，例如锁存器、寄存器、内存。

时钟信号让状态更新具有统一节拍，避免电路在输入变化的瞬间产生不可预测的中间状态。

## 4. 锁存器：最小的可保持状态电路

锁存器（Latch）通过反馈让一个比特保持住。常见的 SR 锁存器有 `S`（Set，置位）和 `R`（Reset，复位）两个控制输入，以及 `Q` 和 `Q̅` 两个互补输出。

以高电平有效的抽象 SR 锁存器为例：

| S | R | Q 的下一状态 | 含义 |
|---:|---:|---:|---|
| 0 | 0 | 保持原值 | 不写入 |
| 1 | 0 | 1 | 置位 |
| 0 | 1 | 0 | 复位 |
| 1 | 1 | 不允许/需按实现判断 | 同时置位和复位 |

不同的 NOR、NAND 实现可能采用低电平有效输入，真值表的有效电平会相应变化。学习时先确认符号上的有效电平，再读表。

### 4.1 为什么能保持

当 `S`、`R` 都撤销后，反馈会把原来的输出状态继续送回输入，使电路停留在原来的稳定状态。这是“记住 1 位”的关键。

## 5. 带边缘触发的锁存器

电平敏感锁存器在允许信号有效的整个时间段都可能响应输入；如果输入在这段时间内变化，输出也会跟着变化。边缘触发结构只在时钟的某个边沿采样输入，能让更新时刻更明确。

- **上升沿触发：**时钟从 0 变 1 时采样；
- **下降沿触发：**时钟从 1 变 0 时采样。

抽象为 D 型存储单元时，可以写成：

```text
在有效时钟边沿：Q_next = D
其他时间：       Q_next = Q
```

### 5.1 时钟的作用

时钟不是“数据”，而是规定状态何时更新的控制信号。多个存储单元使用同一时钟，就能在统一时刻保存一组数据，形成寄存器。

## 6. 8 位寄存器的抽象

一个 8 位寄存器由 8 个 1 位存储单元组成，共享时钟和控制信号：

```text
D[7:0] ──> [bit7][bit6][bit5][bit4][bit3][bit2][bit1][bit0]
                    ↑ 同一个时钟 / 写入控制
Q[7:0] <─────────────┘
```

当写入控制有效时，8 个单元同时保存 `D[7:0]`；控制无效时，输出保持之前的值。

### 6.1 寄存器的典型接口

| 接口 | 作用 |
|---|---|
| `D[7:0]` | 并行输入数据 |
| `Q[7:0]` | 并行输出数据 |
| `CLK` | 时钟 |
| `LOAD/WE` | 是否在时钟边沿写入 |
| `OE` | 是否允许向外部总线输出 |

寄存器的“抽象”隐藏了内部 8 个触发器的细节，上层只需要关心数据、时钟和控制信号。

## 7. 输出使能和三态驱动器

多个寄存器经常共用一条数据总线。若每个寄存器都同时驱动总线，不同输出可能互相冲突。因此需要**输出使能（Output Enable，OE）**。

三态输出除了 0 和 1 外，还有第三种状态 `Z`：高阻态。进入高阻态时，器件相当于暂时断开，不对总线施加 0 或 1。

| OE | 输出状态 |
|---:|---|
| 0 | `Z`，不驱动总线 |
| 1 | 输出寄存器中的数据 |

### 7.1 总线使用规则

- 同一时刻最多允许一个设备把 OE 置为有效；
- 其他设备必须处于高阻态；
- 读取总线的设备可以接收当前值，但不能同时强行输出另一个值。

如果两个输出同时一个驱动 0、一个驱动 1，就会发生总线争用，实物中可能产生大电流，仿真中也会显示冲突或不确定状态。

## 8. 驱动器：让信号可靠地送到总线

课程中的“驱动器”可以理解为位于逻辑模块和总线之间的缓冲/三态驱动单元。它的任务不是改变数据含义，而是：

1. 在允许时把寄存器输出送到总线；
2. 不允许时切换到高阻态；
3. 为总线提供足够的驱动能力，隔离前后级负载。

不要把这里的驱动器和操作系统里的设备驱动程序混淆：这里讨论的是**数字电路中的信号驱动器**。

### 8.1 总线传输示例

```text
寄存器 A --OE_A-->
                    ├── 数据总线 ──> 目标寄存器输入
寄存器 B --OE_B-->
```

当 `OE_A=1`、`OE_B=0` 时，A 可以向总线输出，B 必须断开；下一次传输可以反过来。

## 9. 寄存器输入、输出线路与控制

寄存器通常同时需要“写入控制”和“输出控制”，两者作用不同：

- `LOAD/WE` 决定时钟边沿到来时是否把总线数据写入寄存器；
- `OE` 决定寄存器是否把已保存的数据驱动到总线。

可以用下面的时序理解一次传输：

1. 源寄存器 OE 有效，把数据放到总线；
2. 目标寄存器 LOAD 有效；
3. 在规定的时钟边沿，目标寄存器采样总线并保存；
4. 撤销源寄存器 OE，避免下一次传输冲突。

### 9.1 输入和输出不是同一个开关

一个寄存器可以保持自己的值，同时不对总线输出；也可以在不改变自身值的情况下把数据输出给别的模块。把 `LOAD` 和 `OE` 混为一个信号，会导致“读数据时意外改写寄存器”。

## 10. 并行转串行：移位寄存器的基本原理

**并行输入、串行输出（PISO）**的移位寄存器先一次性装入一组数据，再在连续时钟下逐位送出。

例如装入 `10110010`：

```text
并行装载： [1][0][1][1][0][0][1][0]
第 1 个时钟：输出 bit7，剩余位左移/右移
第 2 个时钟：输出下一位
...
```

具体先输出最高位还是最低位，需要由电路连接和协议约定。

### 10.1 为什么需要移位

并行线路速度高但占用引脚多；串行线路每次只传 1 位，速度可以较低，但能减少连线和引脚。移位寄存器正是两种传输方式之间的转换桥梁。

### 10.2 设计时要写清楚

- 装载时刻；
- 移位方向；
- 第一位输出位置；
- 每个时钟输出一位还是输入一位；
- 数据位数和结束标志。

## 11. 构建一个内存单元格

一个内存单元格首先要能保存 1 位数据。把 8 个单元并排，可以保存 1 个字节；再配上选择和读写控制，就形成可寻址的存储单元。

抽象接口可以写成：

```text
地址选择 ──> 选择器 ──> 某一个存储单元
写入数据 ──> 写入门控
读出数据 <── 输出门控
控制信号：读 / 写 / 时钟
```

### 11.1 读写的基本区别

- **写入：**地址确定目标单元，数据在时钟边沿被保存；
- **读取：**地址确定目标单元，单元中的数据通过输出通路送出；
- **未选中：**单元保持自己的状态，不影响其他单元。

选择信号必须保证同一时刻只有目标单元响应，否则会出现多个单元同时写入或驱动输出的问题。

## 12. 矩阵内存：用行列组织大量单元

如果每一个内存单元都直接连接一根独立选择线，线路数量会迅速增加。矩阵内存把单元排列成行和列，通过行选择、列选择定位目标。

```text
          列0  列1  列2  列3
行0        □    □    □    □
行1        □    □    □    □
行2        □    □    □    □
行3        □    □    □    □
```

### 12.1 地址译码

地址线经过译码器后生成行选择和列选择。例如若行地址有 2 位，就能选择 4 行；列地址也有 2 位，就能选择 4 列，组合后定位 16 个单元中的一个。

译码器的作用是把“二进制地址”转换成“某一条选择线有效”。这使得 CPU 可以用数字地址访问存储单元，而不需要理解内部每根线路。

## 13. 实际内存的结构

实际内存由大量存储单元、地址译码电路、读写控制电路和数据输入输出电路组成。外部通常看到的是统一接口：

| 外部信号 | 作用 |
|---|---|
| 地址总线 | 指定要访问的单元 |
| 数据总线 | 传入写入数据或传出读取数据 |
| 时钟 | 规定同步更新时刻（取决于器件） |
| 读写控制 | 区分读取和写入 |
| 片选/使能 | 指定当前存储芯片是否响应 |

从结构上看，寄存器和内存都能保存数据，但用途不同：寄存器容量小、离运算单元近、访问快；内存容量大、通过地址选择单元，访问通常需要更多译码和控制。

本篇先建立“单元格—矩阵—外部接口”的模型，后续再把寄存器、内存和 CPU 数据总线连接起来。

## 14. 从寄存器到内存的层次关系

可以按容量和访问方式建立一个抽象层次：

```text
1 位锁存器
   ↓ 并排组合
8 位寄存器
   ↓ 增加地址选择
多个字节组成的内存
   ↓ 连接地址总线、数据总线和控制器
计算机的存储系统
```

每一层都由下一层的模块组合而成：

- 锁存器解决“保持一个状态”；
- 寄存器解决“同时保存一组位”；
- 矩阵内存解决“用地址找到大量数据”；
- 存储系统解决“让 CPU 按控制信号读写这些数据”。

这体现了计算机设计中的模块化：上层使用抽象接口，下层负责具体门电路和时序。

## 15. 常见问题与排错方法

### 15.1 锁存器问题

- 把锁存器当成组合门：检查是否存在反馈和保持状态；
- 忽略有效电平：确认 SR 输入是高有效还是低有效；
- 同时激活 Set 和 Reset：避免进入不允许状态。

### 15.2 时钟和寄存器问题

- 在错误的时刻观察输出：先确认是电平敏感还是边沿触发；
- `LOAD` 无效却期待寄存器改变：写入必须满足控制信号和时钟条件；
- 总线数据不对：检查源寄存器 OE、目标寄存器 LOAD 和时钟顺序。

### 15.3 总线和内存问题

- 两个驱动器同时输出：检查 OE，确保其他设备为高阻态；
- 地址变化但数据不变：检查译码器、片选和读写控制；
- 数据写到错误单元：沿地址线逐级检查行列选择；
- 按键或外部输入飘忽：增加合适的上拉或下拉电阻。

## 16. 一页复习表

| 知识点 | 关键结论 |
|---|---|
| 组合逻辑 | 输出只由当前输入决定 |
| 时序逻辑 | 输出还依赖过去保存的状态 |
| 锁存器 | 用反馈保持 1 位数据 |
| 边沿触发 | 在规定时钟边沿采样并更新 |
| 8 位寄存器 | 8 个 1 位存储单元共享控制 |
| LOAD/WE | 决定是否把输入写入寄存器 |
| OE | 决定是否把寄存器输出到总线 |
| 高阻态 Z | 输出端暂时断开，不驱动 0/1 |
| 移位寄存器 | 在时钟作用下逐位移动数据 |
| 内存单元格 | 保存数据并接受地址选择 |
| 矩阵内存 | 用行列译码减少选择线路 |
| 内存接口 | 地址总线、数据总线、读写和使能控制 |

## 17. 自测题与参考答案

1. 组合逻辑和时序逻辑的根本区别是什么？
2. SR 锁存器中的 S、R 分别有什么作用？为什么要避免某些同时有效的组合？
3. 边沿触发存储单元为什么比直接使用电平敏感锁存器更容易统一时序？
4. 8 位寄存器由什么组成？
5. 为什么多个寄存器不能同时驱动同一条总线？OE 和高阻态如何解决？
6. 并行转串行时，为什么必须规定移位方向和首位输出位置？
7. 地址译码器在矩阵内存中完成什么工作？
8. 寄存器和内存的主要差别是什么？

### 参考答案

1. 组合逻辑只看当前输入；时序逻辑还保存并使用过去状态。
2. S 置位、R 复位；同时有效可能产生冲突或不允许状态。
3. 它把更新限制在明确的时钟边沿，减少输入在一个时间窗口内变化造成的歧义。
4. 由 8 个 1 位存储单元组成，并共享时钟、写入和输出控制。
5. 同时驱动 0 和 1 会造成总线争用；OE 无效时让输出进入高阻态。
6. 否则接收端无法判断每个时钟送出的位对应原数据的哪一位。
7. 把二进制地址转换为某一条行选择线和列选择线，从而定位目标单元。
8. 寄存器容量小、靠近运算单元、控制直接；内存由大量单元和译码结构组成，通过地址访问。

## 18. 与原课程资料对应关系

| 本篇内容 | 视频 | 主题对应 |
|---|---|---|
| 计算器与计算机 | 41 | 为什么需要可保存状态的计算机 |
| 电信号存储 | 42 | 从电信号到稳定状态 |
| 锁存器 | 43 | 1 位状态保持 |
| 边缘触发锁存器 | 44 | 时钟边沿与同步更新 |
| 8 位寄存器 | 45 | 多个存储位的抽象 |
| 输出使能、驱动器 | 46—47 | 三态输出与总线连接 |
| 寄存器输入输出线路 | 48 | LOAD、OE 和数据传输 |
| 并行转串行 | 49 | 移位寄存器与逐位传输 |
| 内存单元格 | 50 | 1 个可寻址存储单元 |
| 矩阵内存 | 51 | 行列组织和地址译码 |
| 实际内存结构 | 52 | 存储芯片的外部接口与层次 |

**资料来源：**《自己动手做台计算机》课程，百度网盘分享链接：`https://pan.baidu.com/s/1RLiC8gC1sXTu7EgkQs9oHw?pwd=6633`，提取码：`6633`。本篇按视频 41—52 整理。